In [ ]:
import os
import json
import torch
import random
import numpy as np
import pandas as pd
from tqdm import tqdm
from huggingface_hub import login
from sklearn.metrics import accuracy_score, precision_score, recall_score, ndcg_score
from sentence_transformers import SentenceTransformer, InputExample, losses, evaluation
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoConfig

login("your_huggingface_auth_token_here")

os.environ["WANDB_DISABLED"] = "true"

In [ ]:
def truncate_text(text, tokenizer, max_length=256):
    if not text or not text.strip():
        return ""
    
    try:
        safe_max_length = max(1, max_length - 3)
        
        tokens = tokenizer.encode(
            text, 
            add_special_tokens=False, 
            max_length=safe_max_length, 
            truncation=True
        )
        
        truncated_text = tokenizer.decode(tokens, skip_special_tokens=True)
        
        return truncated_text.strip()
    
    except Exception as e:
        print(f"⚠️ Error truncating text: {e}")
        char_limit = max_length * 4
        return text[:char_limit].strip()

def check_model_config(model_name):
    try:
        config = AutoConfig.from_pretrained(model_name, trust_remote_code=True)
        print(f"📋 MODEL CONFIG:")
        print(f"   Max position embeddings: {getattr(config, 'max_position_embeddings', 'Unknown')}")
        print(f"   Vocab size: {getattr(config, 'vocab_size', 'Unknown')}")
        print(f"   Hidden size: {getattr(config, 'hidden_size', 'Unknown')}")
        print(f"   Model type: {getattr(config, 'model_type', 'Unknown')}")
        return config
    except Exception as e:
        print(f"⚠️ Could not load model config: {e}")
        return None

def debug_data_stats(corpus, queries, test_queries):
    print(f"\n📊 DATA STATISTICS:")

    corpus_lengths = [len(doc['text']) for doc in corpus if doc['text']]
    if corpus_lengths:
        print(f"📚 CORPUS:")
        print(f"   📄 Documents: {len(corpus)}")
        print(f"   📏 Avg length: {np.mean(corpus_lengths):.1f} chars")
        print(f"   📏 Max length: {max(corpus_lengths)} chars")
        print(f"   📏 Min length: {min(corpus_lengths)} chars")
        
        long_docs = [(i, len(doc['text'])) for i, doc in enumerate(corpus) if len(doc['text']) > 5000]
        if long_docs:
            print(f"   ⚠️ Very long documents (>5000 chars): {len(long_docs)}")
    
    query_lengths = [len(q['text']) for q in queries if q['text']]
    if query_lengths:
        print(f"❓ TRAINING QUERIES:")
        print(f"   📄 Queries: {len(queries)}")
        print(f"   📏 Avg length: {np.mean(query_lengths):.1f} chars")
        print(f"   📏 Max length: {max(query_lengths)} chars")

    if test_queries:
        test_query_lengths = [len(q['text']) for q in test_queries if q['text']]
        if test_query_lengths:
            print(f"🧪 TEST QUERIES:")
            print(f"   📄 Queries: {len(test_queries)}")
            print(f"   📏 Avg length: {np.mean(test_query_lengths):.1f} chars")
            print(f"   📏 Max length: {max(test_query_lengths)} chars")
    
    empty_corpus = sum(1 for doc in corpus if not doc['text'] or not doc['text'].strip())
    empty_queries = sum(1 for q in queries if not q['text'] or not q['text'].strip())
    empty_test = sum(1 for q in test_queries if not q['text'] or not q['text'].strip()) if test_queries else 0
    
    if empty_corpus > 0:
        print(f"   ⚠️ Empty corpus documents: {empty_corpus}")
    if empty_queries > 0:
        print(f"   ⚠️ Empty training queries: {empty_queries}")
    if empty_test > 0:
        print(f"   ⚠️ Empty test queries: {empty_test}")
    
    print("=" * 50)

# =============================================================================
# DATA LOADING FUNCTIONS
# =============================================================================

def load_law_corpus(corpus_path, tokenizer=None, truncate_max_length=False, max_seq_length=256):
    """
    Load law corpus từ alqac_2025_law_preview.json
    """
    print(f"📚 Loading law corpus from {corpus_path}...")
    with open(corpus_path, 'r', encoding='utf-8') as f:
        corpus_data = json.load(f)
    
    corpus = []
    
    for law in corpus_data:
        law_id = law.get("id", "unknown_law")
        
        if "articles" in law:
            for article in law["articles"]:
                article_id = article.get("id", "unknown_article")
                content = article.get("text", "")
                
                doc_id = f"{law_id}::{article_id}"
                
                if content.strip():
                    if truncate_max_length and tokenizer:
                        content = truncate_text(content.strip(), tokenizer, max_seq_length)
                    else:
                        content = content.strip()
                    
                    corpus.append({
                        "id": doc_id,
                        "text": content,
                        "law_id": str(law_id),
                        "article_id": str(article_id)
                    })
        else:
            print(f"⚠️ Warning: Law {law_id} has no articles field")
    
    print(f"✅ Loaded {len(corpus)} law articles")
    
    if truncate_max_length:
        print(f"✂️ Text truncation enabled: max {max_seq_length} tokens")
    
    if corpus:
        print(f"📋 Sample document: {corpus[0]['id']}")
        print(f"📄 Sample text: {corpus[0]['text'][:100]}...")
    
    return corpus

def load_training_queries(queries_path, corpus, tokenizer=None, truncate_max_length=False, max_seq_length=256):
    print(f"❓ Loading training queries from {queries_path}...")
    with open(queries_path, 'r', encoding='utf-8') as f:
        queries_data = json.load(f)
    
    queries = []
    qrels = []
    
    article_to_doc_id = {}
    for doc in corpus:
        if "::" in doc['id']:
            law_id_from_doc, article_id_from_doc = doc['id'].split("::", 1)
            key = (law_id_from_doc, article_id_from_doc)
            article_to_doc_id[key] = doc['id']
    
    print(f"🗂️ Created mapping for {len(article_to_doc_id)} articles")
    
    for item in queries_data:
        query_id = item.get('question_id', item.get('id', 'unknown_query'))
        query_text = item.get('text', item.get('question', ''))
        
        if truncate_max_length and tokenizer and query_text:
            query_text = truncate_text(query_text, tokenizer, max_seq_length)
        
        queries.append({
            "id": query_id,
            "text": query_text
        })
        
        if 'relevant_articles' in item:
            for rel_article in item['relevant_articles']:
                law_id = rel_article['law_id']
                article_id = rel_article['article_id']
                
                key = (law_id, article_id)
                if key in article_to_doc_id:
                    doc_id = article_to_doc_id[key]
                    qrels.append({
                        "query_id": query_id,
                        "doc_id": doc_id,
                        "score": 1
                    })
                else:
                    print(f"⚠️ Warning: Article not found: {law_id} - {article_id}")
    
    print(f"✅ Loaded {len(queries)} training queries with {len(qrels)} relevance judgments")
    return queries, qrels

def load_test_queries(test_path, corpus, tokenizer=None, truncate_max_length=False, max_seq_length=256):
    print(f"🧪 Loading test queries from {test_path}...")
    with open(test_path, 'r', encoding='utf-8') as f:
        test_data = json.load(f)
    
    test_queries = []
    test_qrels = []
    
    article_to_doc_id = {}
    for doc in corpus:
        if "::" in doc['id']:
            law_id_from_doc, article_id_from_doc = doc['id'].split("::", 1)
            key = (law_id_from_doc, article_id_from_doc)
            article_to_doc_id[key] = doc['id']
    
    for item in test_data:
        query_id = item.get('question_id', item.get('id', 'unknown_test_query'))
        query_text = item.get('text', item.get('question', ''))
        
        if truncate_max_length and tokenizer and query_text:
            query_text = truncate_text(query_text, tokenizer, max_seq_length)
        
        test_queries.append({
            "id": query_id,
            "text": query_text
        })
        
        if 'relevant_articles' in item:
            for rel_article in item['relevant_articles']:
                law_id = rel_article['law_id']
                article_id = rel_article['article_id']
                
                key = (law_id, article_id)
                if key in article_to_doc_id:
                    doc_id = article_to_doc_id[key]
                    test_qrels.append({
                        "query_id": query_id,
                        "doc_id": doc_id,
                        "score": 1
                    })
    
    print(f"✅ Loaded {len(test_queries)} test queries with {len(test_qrels)} relevance judgments")
    return test_queries, test_qrels

# =============================================================================
# DATA SPLITTING FUNCTIONS
# =============================================================================

def create_data_splits(queries, n_validation=100):
    """
    Chia data thành train/validation (2-way split)
    """
    n_queries = len(queries)
    n_validation = min(n_validation, n_queries // 2)
    
    if n_queries <= 100:
        print(f"⚠️ Warning: Total queries ({n_queries}) <= 100. Using 50% for validation.")
        n_validation = n_queries // 2
    
    validation_indices = list(range(n_validation))
    train_indices = list(range(n_validation, n_queries))
    
    splits = {
        "train": [queries[i]["id"] for i in train_indices],
        "validation": [queries[i]["id"] for i in validation_indices]
    }
    
    print(f"📊 DATA SPLIT:")
    print(f"   🏋️ Training: {len(train_indices)} samples")
    print(f"   ✅ Validation: {len(validation_indices)} samples (first {n_validation})")
    print(f"   📈 Hard negative mining will use {len(train_indices)} training samples")
    
    return splits

def create_three_way_split(queries, n_test=100, n_validation=100, shuffle=True, random_seed=42):
    n_queries = len(queries)
    
    if n_queries < (n_test + n_validation):
        print(f"⚠️ Warning: Not enough data. Total: {n_queries}, Requested: {n_test + n_validation}")
        ratio = n_queries / (n_test + n_validation)
        n_test = int(n_test * ratio)
        n_validation = int(n_validation * ratio)
        print(f"   Adjusted: test={n_test}, val={n_validation}")
    
    indices = list(range(n_queries))
    
    if shuffle:
        random.seed(random_seed)
        random.shuffle(indices)
        print(f"🔀 Data shuffled with seed: {random_seed}")
    
    test_indices = indices[:n_test]
    val_indices = indices[n_test:n_test + n_validation]
    train_indices = indices[n_test + n_validation:]
    
    splits = {
        "train": [queries[i]["id"] for i in train_indices],
        "validation": [queries[i]["id"] for i in val_indices],
        "test": [queries[i]["id"] for i in test_indices]
    }
    
    print(f"📊 THREE-WAY DATA SPLIT:")
    print(f"   🏋️ Training: {len(train_indices)} samples")
    print(f"   ✅ Validation: {len(val_indices)} samples")
    print(f"   🧪 Test: {len(test_indices)} samples")
    print(f"   📈 Hard negative mining will use {len(train_indices)} training samples")
    
    return splits

def create_qrels_for_split(original_qrels, split_query_ids, split_name):
    split_qrels = []
    for qrel in original_qrels:
        if qrel["query_id"] in split_query_ids:
            split_qrels.append(qrel)
    
    print(f"   📋 {split_name} qrels: {len(split_qrels)} relevance judgments")
    return split_qrels

# =============================================================================
# DATA SAVING FUNCTIONS
# =============================================================================

def save_processed_data(corpus, queries, qrels, test_queries, test_qrels, splits, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    
    with open(os.path.join(output_dir, "corpus.jsonl"), "w", encoding="utf-8") as f:
        for doc in corpus:
            f.write(json.dumps(doc, ensure_ascii=False) + "\n")
    
    with open(os.path.join(output_dir, "training_queries.jsonl"), "w", encoding="utf-8") as f:
        for query in queries:
            f.write(json.dumps(query, ensure_ascii=False) + "\n")
    
    with open(os.path.join(output_dir, "test_queries.jsonl"), "w", encoding="utf-8") as f:
        for query in test_queries:
            f.write(json.dumps(query, ensure_ascii=False) + "\n")

    with open(os.path.join(output_dir, "training_qrels.jsonl"), "w", encoding="utf-8") as f:
        for qrel in qrels:
            f.write(json.dumps(qrel, ensure_ascii=False) + "\n")
    
    with open(os.path.join(output_dir, "test_qrels.jsonl"), "w", encoding="utf-8") as f:
        for qrel in test_qrels:
            f.write(json.dumps(qrel, ensure_ascii=False) + "\n")

    with open(os.path.join(output_dir, "data_splits.json"), "w", encoding="utf-8") as f:
        json.dump(splits, f, ensure_ascii=False, indent=2)
    
    print(f"💾 Saved processed data to {output_dir}")

def save_processed_data_three_way(corpus, queries, qrels, splits, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    
    with open(os.path.join(output_dir, "corpus.jsonl"), "w", encoding="utf-8") as f:
        for doc in corpus:
            f.write(json.dumps(doc, ensure_ascii=False) + "\n")
    
    train_queries = [q for q in queries if q["id"] in splits["train"]]
    val_queries = [q for q in queries if q["id"] in splits["validation"]]
    test_queries = [q for q in queries if q["id"] in splits["test"]]
    
    with open(os.path.join(output_dir, "train_queries.jsonl"), "w", encoding="utf-8") as f:
        for query in train_queries:
            f.write(json.dumps(query, ensure_ascii=False) + "\n")
    
    with open(os.path.join(output_dir, "val_queries.jsonl"), "w", encoding="utf-8") as f:
        for query in val_queries:
            f.write(json.dumps(query, ensure_ascii=False) + "\n")
    
    with open(os.path.join(output_dir, "test_queries.jsonl"), "w", encoding="utf-8") as f:
        for query in test_queries:
            f.write(json.dumps(query, ensure_ascii=False) + "\n")
    
    train_qrels = create_qrels_for_split(qrels, splits["train"], "Train")
    val_qrels = create_qrels_for_split(qrels, splits["validation"], "Validation")
    test_qrels = create_qrels_for_split(qrels, splits["test"], "Test")
    
    with open(os.path.join(output_dir, "train_qrels.jsonl"), "w", encoding="utf-8") as f:
        for qrel in train_qrels:
            f.write(json.dumps(qrel, ensure_ascii=False) + "\n")
    
    with open(os.path.join(output_dir, "val_qrels.jsonl"), "w", encoding="utf-8") as f:
        for qrel in val_qrels:
            f.write(json.dumps(qrel, ensure_ascii=False) + "\n")
    
    with open(os.path.join(output_dir, "test_qrels.jsonl"), "w", encoding="utf-8") as f:
        for qrel in test_qrels:
            f.write(json.dumps(qrel, ensure_ascii=False) + "\n")
    
    with open(os.path.join(output_dir, "three_way_splits.json"), "w", encoding="utf-8") as f:
        splits_info = {
            "splits": splits,
            "train_size": len(splits["train"]),
            "val_size": len(splits["validation"]),
            "test_size": len(splits["test"]),
            "total_size": len(queries)
        }
        json.dump(splits_info, f, ensure_ascii=False, indent=2)
    
    print(f"💾 Saved three-way split data to {output_dir}")
    return train_qrels, val_qrels, test_qrels

# =============================================================================
# HARD NEGATIVE MINING
# =============================================================================

def generate_hard_negatives(corpus, queries, qrels, splits, n_hard_negatives=5, model_name="FacebookAI/xlm-roberta-large"):
    print(f"⚡ Generating hard negatives using {model_name}...")
    model = SentenceTransformer(model_name, trust_remote_code=True)
    
    if hasattr(model, 'max_seq_length'):
        print(f"📏 Model max_seq_length: {model.max_seq_length}")
    else:
        model.max_seq_length = 256
        print(f"📏 Set model max_seq_length to: {model.max_seq_length}")
    
    corpus_texts = []
    corpus_ids = []
    
    for item in corpus:
        text = item["text"]
        if text and text.strip() and len(text.strip()) > 0:
            if len(text) > 10000:
                text = text[:10000]
            corpus_texts.append(text.strip())
            corpus_ids.append(item["id"])
        else:
            print(f"⚠️ Skipping empty document: {item['id']}")
    
    print(f"🔢 Encoding {len(corpus_texts)} documents (filtered from {len(corpus)})...")
    
    try:
        corpus_embeddings = model.encode(
            corpus_texts, 
            convert_to_tensor=True, 
            show_progress_bar=True,
            batch_size=32,
            device=model.device if hasattr(model, 'device') else None
        )
    except Exception as e:
        print(f"❌ Error during encoding: {e}")
        print("🔧 Trying with smaller batch size and CPU fallback...")
        corpus_embeddings = model.encode(
            corpus_texts, 
            convert_to_tensor=True, 
            show_progress_bar=True,
            batch_size=1,
            device='cpu'
        )
    
    query_to_positives = {}
    for qrel in qrels:
        if qrel["query_id"] not in query_to_positives:
            query_to_positives[qrel["query_id"]] = []
        query_to_positives[qrel["query_id"]].append(qrel["doc_id"])
    
    qid_to_query = {item["id"]: item["text"] for item in queries}
    
    train_examples = []
    train_queries = splits["train"]
    
    print(f"🔍 Mining hard negatives for {len(train_queries)} training queries...")
    
    for query_id in tqdm(train_queries, desc="Hard negative mining"):
        if query_id not in query_to_positives or query_id not in qid_to_query:
            continue
            
        query_text = qid_to_query[query_id]
        positive_ids = query_to_positives[query_id]
        
        positive_texts = []
        for doc_id in positive_ids:
            if doc_id in corpus_ids:
                idx = corpus_ids.index(doc_id)
                positive_texts.append(corpus_texts[idx])
        
        if not positive_texts:
            continue
        
        if not query_text or not query_text.strip():
            print(f"⚠️ Skipping empty query: {query_id}")
            continue
            
        if len(query_text) > 1000:
            query_text = query_text[:1000]
            
        try:
            query_embedding = model.encode(query_text, convert_to_tensor=True)
        except Exception as e:
            print(f"⚠️ Error encoding query {query_id}: {e}")
            continue
        
        cos_scores = torch.nn.functional.cosine_similarity(query_embedding, corpus_embeddings)
        
        hard_negative_indices = []
        cos_scores_np = cos_scores.cpu().numpy()
        sorted_indices = np.argsort(-cos_scores_np)
        
        for idx in sorted_indices:
            doc_id = corpus_ids[idx]
            if doc_id not in positive_ids:
                hard_negative_indices.append(idx)
                if len(hard_negative_indices) >= n_hard_negatives:
                    break
        
        hard_negative_texts = [corpus_texts[idx] for idx in hard_negative_indices]
        
        for pos_text in positive_texts:
            train_examples.append(InputExample(
                texts=[query_text, pos_text] + hard_negative_texts[:3] 
            ))
    
    print(f"🎲 Adding random negatives for diversity...")
    for query_id in train_queries:
        if query_id not in query_to_positives or query_id not in qid_to_query:
            continue
            
        query_text = qid_to_query[query_id]
        positive_ids = query_to_positives[query_id]
        
        for pos_id in positive_ids:
            if pos_id not in corpus_ids:
                continue
                
            pos_idx = corpus_ids.index(pos_id)
            pos_text = corpus_texts[pos_idx]
            
            random_neg_indices = []
            while len(random_neg_indices) < 3:
                idx = random.randint(0, len(corpus_texts) - 1)
                if corpus_ids[idx] not in positive_ids and idx not in random_neg_indices:
                    random_neg_indices.append(idx)
            
            random_neg_texts = [corpus_texts[idx] for idx in random_neg_indices]
            
            train_examples.append(InputExample(
                texts=[query_text, pos_text] + random_neg_texts
            ))
    
    print(f"✅ Generated {len(train_examples)} training examples with hard negatives")
    return train_examples

# =============================================================================
# EVALUATION SETUP
# =============================================================================

def setup_validation_evaluator(corpus, queries, qrels, splits):
    validation_queries = {q["id"]: q["text"] for q in queries if q["id"] in splits["validation"]}
    validation_relevant_docs = {}
    
    for qrel in qrels:
        if qrel["query_id"] in validation_queries:
            if qrel["query_id"] not in validation_relevant_docs:
                validation_relevant_docs[qrel["query_id"]] = []
            validation_relevant_docs[qrel["query_id"]].append(qrel["doc_id"])
    
    corpus_dict = {doc["id"]: doc["text"] for doc in corpus}
    
    print(f"✅ Setup validation evaluator: {len(validation_queries)} queries")
    
    return evaluation.InformationRetrievalEvaluator(
        validation_queries, corpus_dict, validation_relevant_docs, name='validation-eval'
    )

def setup_evaluators_three_way(corpus, queries, qrels, splits):

    validation_queries = {q["id"]: q["text"] for q in queries if q["id"] in splits["validation"]}
    validation_qrels = create_qrels_for_split(qrels, splits["validation"], "Validation")
    validation_relevant_docs = {}
    
    for qrel in validation_qrels:
        if qrel["query_id"] not in validation_relevant_docs:
            validation_relevant_docs[qrel["query_id"]] = []
        validation_relevant_docs[qrel["query_id"]].append(qrel["doc_id"])
    
    test_queries = {q["id"]: q["text"] for q in queries if q["id"] in splits["test"]}
    test_qrels = create_qrels_for_split(qrels, splits["test"], "Test")
    test_relevant_docs = {}
    
    for qrel in test_qrels:
        if qrel["query_id"] not in test_relevant_docs:
            test_relevant_docs[qrel["query_id"]] = []
        test_relevant_docs[qrel["query_id"]].append(qrel["doc_id"])
    
    corpus_dict = {doc["id"]: doc["text"] for doc in corpus}
    
    validation_evaluator = evaluation.InformationRetrievalEvaluator(
        validation_queries, corpus_dict, validation_relevant_docs, name='validation-eval'
    )
    
    test_evaluator = evaluation.InformationRetrievalEvaluator(
        test_queries, corpus_dict, test_relevant_docs, name='test-eval'
    )
    
    print(f"✅ Setup evaluators:")
    print(f"   📊 Validation: {len(validation_queries)} queries")
    print(f"   🧪 Test: {len(test_queries)} queries")
    
    return validation_evaluator, test_evaluator

# =============================================================================
# MODEL TRAINING
# =============================================================================

def fine_tune_model(train_examples, validation_evaluator, output_dir, n_epochs=1, model_name="FacebookAI/xlm-roberta-large", max_seq_length=256):

    print(f"🤖 Loading model: {model_name}")
    model = SentenceTransformer(model_name, trust_remote_code=True)
    
    try:
        if hasattr(model, 'max_seq_length'):
            print(f"📏 Current model max_seq_length: {model.max_seq_length}")
        model.max_seq_length = max_seq_length
        print(f"📏 Set model max_seq_length to: {max_seq_length}")
        
        if hasattr(model, 'tokenizer') and hasattr(model.tokenizer, 'model_max_length'):
            model.tokenizer.model_max_length = max_seq_length
            print(f"📏 Set tokenizer model_max_length to: {max_seq_length}")
            
    except Exception as e:
        print(f"⚠️ Warning setting max_seq_length: {e}")
    
    train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=8)
    
    train_loss = losses.MultipleNegativesRankingLoss(model)
    
    print(f"🏋️ Starting training...")
    print(f"   📊 Training examples: {len(train_examples)}")
    print(f"   🔄 Epochs: {n_epochs}")
    print(f"   📁 Model will be saved to: {os.path.join(output_dir, 'model')}")
    
    try:
        model.fit(
            train_objectives=[(train_dataloader, train_loss)],
            evaluator=validation_evaluator,
            epochs=n_epochs,
            evaluation_steps=1000,
            warmup_steps=100,
            optimizer_params={'lr': 2.5e-5},
            output_path=os.path.join(output_dir, "model"),
            save_best_model=True,
            use_amp=False 
        )
    except Exception as e:
        print(f"❌ Training failed with error: {e}")
        print("🔧 Trying with modified parameters...")
        model.fit(
            train_objectives=[(train_dataloader, train_loss)],
            epochs=1,
            warmup_steps=50,
            optimizer_params={'lr': 2e-5},
            output_path=os.path.join(output_dir, "model"),
            save_best_model=False,
            use_amp=False
        )
    
    print(f"✅ Training completed! Model saved.")
    return model

# =============================================================================
# EVALUATION FUNCTIONS
# =============================================================================

def evaluate_on_test_set(model, corpus, test_queries, test_qrels, output_dir):
    print(f"🧪 Evaluating on test benchmark...")
    
    test_queries_dict = {q["id"]: q["text"] for q in test_queries}
    test_relevant_docs = {}
    
    for qrel in test_qrels:
        if qrel["query_id"] not in test_relevant_docs:
            test_relevant_docs[qrel["query_id"]] = []
        test_relevant_docs[qrel["query_id"]].append(qrel["doc_id"])
    
    corpus_dict = {doc["id"]: doc["text"] for doc in corpus}
    
    test_evaluator = evaluation.InformationRetrievalEvaluator(
        test_queries_dict, corpus_dict, test_relevant_docs, name='test-benchmark'
    )
    
    test_results = test_evaluator(model)
    
    with open(os.path.join(output_dir, "test_benchmark_results.json"), "w", encoding="utf-8") as f:
        json.dump(test_results, f, indent=4, ensure_ascii=False)
    
    print(f"📊 TEST BENCHMARK RESULTS:")
    for metric, value in test_results.items():
        print(f"   {metric}: {value:.4f}")
    
    return test_results

def evaluate_on_custom_test_set(model, test_evaluator, output_dir, test_name="custom_test"):
    print(f"🧪 Evaluating on {test_name}...")
    
    test_results = test_evaluator(model)
    
    results_file = os.path.join(output_dir, f"{test_name}_results.json")
    with open(results_file, "w", encoding="utf-8") as f:
        json.dump(test_results, f, indent=4, ensure_ascii=False)
    
    print(f"📊 {test_name.upper()} RESULTS:")
    for metric, value in test_results.items():
        print(f"   {metric}: {value:.4f}")
    
    return test_results

In [ ]:
# =============================================================================
# MAIN PIPELINES
# =============================================================================

def main_pipeline(corpus_path, queries_path, test_path, output_dir, 
                 model_name="FacebookAI/xlm-roberta-large", n_epochs=1,
                 truncate_max_length=False, max_seq_length=256):
    print("🚀 ALQAC 2025 IR FINE-TUNING PIPELINE (ORIGINAL)")
    print("=" * 60)
    print(f"📚 Law Corpus: {corpus_path}")
    print(f"❓ Training Queries: {queries_path}")
    print(f"🧪 Test Benchmark: {test_path}")
    print(f"🤖 Model: {model_name}")
    print(f"🔄 Epochs: {n_epochs}")
    print(f"✂️ Truncate Max Length: {truncate_max_length}")
    if truncate_max_length:
        print(f"📏 Max Sequence Length: {max_seq_length} tokens")
    print("=" * 60)
    
    tokenizer = None
    if truncate_max_length:
        print(f"🔤 Loading tokenizer for text truncation...")
        try:
            tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
            print(f"✅ Tokenizer loaded successfully")
            
            config = check_model_config(model_name)
            if config and hasattr(config, 'max_position_embeddings'):
                model_max_pos = config.max_position_embeddings
                if max_seq_length > model_max_pos:
                    print(f"⚠️ Warning: Requested max_seq_length ({max_seq_length}) > model max_position_embeddings ({model_max_pos})")
                    print(f"   Adjusting max_seq_length to {model_max_pos - 10}")
                    max_seq_length = model_max_pos - 10
                    
        except Exception as e:
            print(f"⚠️ Warning: Could not load tokenizer: {e}")
            print(f"   Falling back to no truncation")
            truncate_max_length = False
    
    print("\n📋 STEP 1: Loading data...")
    corpus = load_law_corpus(corpus_path, tokenizer, truncate_max_length, max_seq_length)
    queries, qrels = load_training_queries(queries_path, corpus, tokenizer, truncate_max_length, max_seq_length)
    test_queries, test_qrels = load_test_queries(test_path, corpus, tokenizer, truncate_max_length, max_seq_length)
    
    debug_data_stats(corpus, queries, test_queries)
    
    print("\n📊 STEP 2: Creating data splits...")
    splits = create_data_splits(queries, n_validation=100)
    
    print("\n💾 STEP 3: Saving processed data...")
    save_processed_data(corpus, queries, qrels, test_queries, test_qrels, splits, output_dir)
    
    print("\n⚡ STEP 4: Generating hard negatives...")
    train_examples = generate_hard_negatives(corpus, queries, qrels, splits, model_name=model_name)
    
    print("\n✅ STEP 5: Setting up validation evaluator...")
    validation_evaluator = setup_validation_evaluator(corpus, queries, qrels, splits)
    
    print("\n🏋️ STEP 6: Fine-tuning model...")
    model = fine_tune_model(train_examples, validation_evaluator, output_dir, n_epochs, model_name, max_seq_length)
    
    print("\n🧪 STEP 7: Test benchmark evaluation...")
    test_results = evaluate_on_test_set(model, corpus, test_queries, test_qrels, output_dir)
    
    print("\n🎉 PIPELINE COMPLETED SUCCESSFULLY!")
    print(f"📁 All results saved to: {output_dir}")
    if truncate_max_length:
        print(f"✂️ Text truncation was applied with max length: {max_seq_length} tokens")
    print("=" * 60)
    
    return {
        "model": model,
        "test_results": test_results,
        "data_info": {
            "corpus_size": len(corpus),
            "train_queries": len(splits["train"]),
            "validation_queries": len(splits["validation"]),
            "test_queries": len(test_queries),
            "training_examples": len(train_examples),
            "truncation_enabled": truncate_max_length,
            "max_seq_length": max_seq_length if truncate_max_length else None
        }
    }

In [ ]:
def main_pipeline_three_way_split(corpus_path, queries_path, output_dir, 
                                  model_name="FacebookAI/xlm-roberta-large", n_epochs=1,
                                  truncate_max_length=False, max_seq_length=256,
                                  n_test=100, n_validation=100, shuffle=True, random_seed=42):
    """
    Main pipeline với three-way split: train/validation/test từ cùng một file
    """
    print("🚀 ALQAC 2025 IR FINE-TUNING PIPELINE (THREE-WAY SPLIT)")
    print("=" * 70)
    print(f"📚 Law Corpus: {corpus_path}")
    print(f"❓ Queries File: {queries_path}")
    print(f"🤖 Model: {model_name}")
    print(f"🔄 Epochs: {n_epochs}")
    print(f"✂️ Truncate Max Length: {truncate_max_length}")
    if truncate_max_length:
        print(f"📏 Max Sequence Length: {max_seq_length} tokens")
    print(f"📊 Split sizes: Test={n_test}, Val={n_validation}, Train=remainder")
    print(f"🔀 Shuffle: {shuffle} (seed: {random_seed})")
    print("=" * 70)
    
    tokenizer = None
    if truncate_max_length:
        print(f"🔤 Loading tokenizer for text truncation...")
        try:
            tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
            print(f"✅ Tokenizer loaded successfully")
            
            config = check_model_config(model_name)
            if config and hasattr(config, 'max_position_embeddings'):
                model_max_pos = config.max_position_embeddings
                if max_seq_length > model_max_pos:
                    print(f"⚠️ Warning: Requested max_seq_length ({max_seq_length}) > model max_position_embeddings ({model_max_pos})")
                    print(f"   Adjusting max_seq_length to {model_max_pos - 10}")
                    max_seq_length = model_max_pos - 10
                    
        except Exception as e:
            print(f"⚠️ Warning: Could not load tokenizer: {e}")
            print(f"   Falling back to no truncation")
            truncate_max_length = False
    
    print("\n📋 STEP 1: Loading data...")
    corpus = load_law_corpus(corpus_path, tokenizer, truncate_max_length, max_seq_length)
    queries, qrels = load_training_queries(queries_path, corpus, tokenizer, truncate_max_length, max_seq_length)
    
    debug_data_stats(corpus, queries, [])

    print("\n📊 STEP 2: Creating three-way data splits...")
    splits = create_three_way_split(queries, n_test, n_validation, shuffle, random_seed)
    
    print("\n💾 STEP 3: Saving processed data...")
    train_qrels, val_qrels, test_qrels = save_processed_data_three_way(corpus, queries, qrels, splits, output_dir)
    
    print("\n⚡ STEP 4: Generating hard negatives...")
    train_examples = generate_hard_negatives(corpus, queries, qrels, splits, model_name=model_name)
    
    print("\n✅ STEP 5: Setting up evaluators...")
    validation_evaluator, test_evaluator = setup_evaluators_three_way(corpus, queries, qrels, splits)
    
    print("\n🏋️ STEP 6: Fine-tuning model...")
    model = fine_tune_model(train_examples, validation_evaluator, output_dir, n_epochs, model_name, max_seq_length)
    
    print("\n📊 STEP 7: Validation evaluation completed during training")
    
    print("\n🧪 STEP 8: Custom test set evaluation...")
    test_results = evaluate_on_custom_test_set(model, test_evaluator, output_dir, "custom_test")
    
    print("\n🎉 THREE-WAY PIPELINE COMPLETED SUCCESSFULLY!")
    print(f"📁 All results saved to: {output_dir}")
    if truncate_max_length:
        print(f"✂️ Text truncation was applied with max length: {max_seq_length} tokens")
    print("=" * 70)
    
    return {
        "model": model,
        "test_results": test_results,
        "data_info": {
            "corpus_size": len(corpus),
            "train_queries": len(splits["train"]),
            "validation_queries": len(splits["validation"]),
            "test_queries": len(splits["test"]),
            "training_examples": len(train_examples),
            "truncation_enabled": truncate_max_length,
            "max_seq_length": max_seq_length if truncate_max_length else None
        },
        "splits": splits
    }

In [ ]:
# =============================================================================
# CONVENIENCE FUNCTIONS
# =============================================================================

def run_three_way_split_only():
    """
    Run three-way split pipeline only
    """
    corpus_path = "./ALQAC_2025/alqac25_law.json"
    queries_path = "./ALQAC_2025/alqac25_train.json"
    output_dir = "./three_way_split/"
    
    model_name = "./models/ViLegalBERT"
    n_epochs = 2
    
    truncate_max_length = True
    max_seq_length = 200 

    print("🚀 RUNNING THREE-WAY SPLIT PIPELINE ONLY")
    print("=" * 60)
    
    results = main_pipeline_three_way_split(
        corpus_path=corpus_path,
        queries_path=queries_path,
        output_dir=output_dir,
        model_name=model_name,
        n_epochs=n_epochs,
        truncate_max_length=truncate_max_length,
        max_seq_length=max_seq_length,
        n_test=100,         
        n_validation=100,   
        shuffle=True,       
        random_seed=42,     
        train_batch_size=8,     
        encoding_batch_size=32  
    )
    
    print(f"\n🎯 THREE-WAY SPLIT RESULTS:")
    print(f"📁 Output saved to: {output_dir}")
    print(f"📊 Splits: Train={len(results['splits']['train'])}, Val={len(results['splits']['validation'])}, Test={len(results['splits']['test'])}")
    print(f"🧪 Test performance: Check {output_dir}custom_test_results.json")
    
    return results


In [ ]:
corpus_path = "./ZALO/zalo_corpus.json"  
queries_path = "./ZALO/zalo_question.json"  
test_path = "./alqac-2022-2025/alqac25_private_test_task2.json"  
output_dir = "./working/"  

model_name = "./models/ViLegalBERT"  
n_epochs = 1  


truncate_max_length = True  
max_seq_length = 256  

print("\n" + "="*50)
print("🚀 RUNNING THREE-WAY SPLIT PIPELINE")
print("="*50)

results_three_way = main_pipeline_three_way_split(
    corpus_path=corpus_path,
    queries_path=queries_path,  
    output_dir=output_dir + "three_way/",
    model_name=model_name,
    n_epochs=n_epochs,
    truncate_max_length=truncate_max_length,
    max_seq_length=max_seq_length,
    n_test=100, # extract 100 samples from `./ZALO/zalo_question.json` (See create_three_way_split function for detail)
    n_validation=100,   # extract 100 samples from `./ZALO/zalo_question.json` (See create_three_way_split function for detail)
    shuffle=True,       
    random_seed=42      
)

print(f"\n📈 COMPARISON SUMMARY:")
print(f"🔹 Original Pipeline:")
print(f"   ✅ Model trained and saved to: {output_dir}original/")
print(f"   📊 Data info: {results_original['data_info']}")

print(f"🔹 Three-way Pipeline:")
print(f"   ✅ Model trained and saved to: {output_dir}three_way/")
print(f"   📊 Data info: {results_three_way['data_info']}")
print(f"   🔀 Splits: Train={len(results_three_way['splits']['train'])}, Val={len(results_three_way['splits']['validation'])}, Test={len(results_three_way['splits']['test'])}")